# Bounded GRU grid model for N=8 trajectories

This notebook trains a categorical next-state model on a uniform grid over $s_z \in [-1,1]$. Unlike a Gaussian increment model, every generated state is inside the physical interval by construction.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split

# Works when Jupyter starts from either the project root or att_N8/.
cwd = Path.cwd().resolve()
if (cwd / "att_N8").is_dir():
    project_dir = cwd
    experiment_dir = cwd / "att_N8"
elif cwd.name == "att_N8":
    project_dir = cwd.parent
    experiment_dir = cwd
else:
    raise RuntimeError("Start Jupyter from the project root or att_N8/.")

sys.path.insert(0, str(experiment_dir))

from dataset import SingleNNextStateDataset
from model import TrajectoryGRUGrid, interpolated_grid_nll
from train_grid_model import evaluate, generate_trajectories, rollout_summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project directory:", project_dir)
print("Using:", device)

## Load trajectories

Change `data_dir` here if the data is stored elsewhere. Each `trajectory_*.npy` file is expected to contain one one-dimensional trajectory.

In [ ]:
candidate_data_dirs = [
    project_dir / "generate_traj" / "data" / "sz_N8",
    project_dir / "GenerateTraj" / "data" / "sz_N8",
]
data_dir = next((path for path in candidate_data_dirs if path.is_dir()), candidate_data_dirs[0])
files = sorted(data_dir.glob("trajectory_*.npy"))

if not files:
    raise FileNotFoundError(f"No trajectory_*.npy files found in {data_dir.resolve()}.")

sz = np.stack([np.load(path) for path in files]).astype(np.float32)

print("Number of trajectories:", len(sz))
print("Array shape:", sz.shape)
print("Value range:", sz.min(), sz.max())
print("Initial-state range:", sz[:, 0].min(), sz[:, 0].max())

## Dataset and trajectory-level split

The input at each step is `[current_sz, normalized_time]`; the target is the next state itself. The corrected time feature uses the source-state times $t_0,\ldots,t_{T-2}$.

In [ ]:
state_min = -1.0
state_max = 1.0
time_end = 1.0
seed = 42

dataset = SingleNNextStateDataset(
    sz,
    time_end=time_end,
    state_min=state_min,
    state_max=state_max,
)

n_total = len(dataset)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)
n_test = n_total - n_train - n_val

split_generator = torch.Generator().manual_seed(seed)
train_set, val_set, test_set = random_split(
    dataset,
    [n_train, n_val, n_test],
    generator=split_generator,
)

batch_size = 64
pin_memory = device.type == "cuda"
shuffle_generator = torch.Generator().manual_seed(seed)

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    generator=shuffle_generator,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=pin_memory,
)

features, next_sz = next(iter(train_loader))
print("Input batch:", features.shape)
print("Target batch:", next_sz.shape)
print("Split sizes:", n_train, n_val, n_test)

## Create the bounded grid model

With 401 points on `[-1,1]`, the state spacing is 0.005. Use 201 points for spacing 0.01 and lower memory use.

In [ ]:
torch.manual_seed(seed)

model_config = {
    "input_size": 2,
    "hidden_size": 128,
    "num_layers": 2,
    "grid_size": 401,
    "dropout": 0.1,
    "state_min": state_min,
    "state_max": state_max,
}

model = TrajectoryGRUGrid(**model_config).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-5,
)

grid_spacing = (state_max - state_min) / (model.grid_size - 1)
print(model)
print("Grid spacing:", grid_spacing)
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

## Train and save the best validation checkpoint

`interpolated_grid_nll` distributes each continuous target between its two neighboring grid points. Grid NLL values are categorical losses and should not be compared numerically with the old Gaussian-density NLL.

In [ ]:
epochs = 100
best_val_loss = float("inf")
train_losses = []
val_losses = []

checkpoint_path = experiment_dir / "n8_grugrid_best.pt"

for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0
    total_targets = 0

    for features, next_sz in train_loader:
        features = features.to(device, non_blocking=True)
        next_sz = next_sz.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(features)
        loss = interpolated_grid_nll(
            next_sz,
            logits,
            state_min=model.state_min,
            state_max=model.state_max,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        target_count = next_sz.numel()
        total_train_loss += loss.item() * target_count
        total_targets += target_count

    train_loss = total_train_loss / total_targets
    val_loss = evaluate(model, val_loader, device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch": epoch,
                "model_type": "TrajectoryGRUGrid",
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_grid_nll": val_loss,
                "model_config": model_config,
                "time_end": time_end,
                "seed": seed,
                "split_indices": {
                    "train": list(train_set.indices),
                    "validation": list(val_set.indices),
                    "test": list(test_set.indices),
                },
                "source_files": [path.name for path in files],
            },
            checkpoint_path,
        )

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d}/{epochs} | "
            f"train grid NLL: {train_loss:.5f} | "
            f"validation grid NLL: {val_loss:.5f}"
        )

print(f"Best validation grid NLL: {best_val_loss:.5f}")
print("Saved checkpoint:", checkpoint_path)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="validation")
plt.xlabel("Epoch")
plt.ylabel("Categorical grid NLL")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Reload the best model and evaluate it

The test grid NLL is still a teacher-forced one-step metric. The free-running plots and statistics below are needed to evaluate trajectory generation.

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)

# Reconstructing from the saved configuration avoids hidden default mismatches.
model = TrajectoryGRUGrid(**checkpoint["model_config"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss = evaluate(model, test_loader, device)
print("Best checkpoint epoch:", checkpoint["epoch"])
print(f"Test grid NLL: {test_loss:.5f}")

## Generate one trajectory

`temperature=1.0` samples the learned categorical distribution. Values below 1 sharpen it; values above 1 increase diversity.

In [ ]:
reference_index = test_set.indices[0]
reference = torch.from_numpy(sz[reference_index])
sample_generator = torch.Generator(device=device).manual_seed(4)

generated = generate_trajectories(
    model,
    initial_sz=reference[0:1],
    n_time_points=reference.numel(),
    time_end=time_end,
    temperature=1.0,
    generator=sample_generator,
).cpu().numpy()[0]

physical_time = np.linspace(0.0, 70.0, len(generated))

print("Generated range:", generated.min(), generated.max())
assert generated.min() >= state_min and generated.max() <= state_max

plt.figure(figsize=(12, 3.5))
plt.plot(physical_time, generated, "b-o", ms=3, label="generated")
plt.plot(
    physical_time,
    reference.numpy(),
    "k-o",
    ms=3,
    alpha=0.6,
    label="one held-out exact trajectory",
)
plt.axhline(state_min, color="red", ls=":", alpha=0.5)
plt.axhline(state_max, color="red", ls=":", alpha=0.5)
plt.xlabel(r"Time $t$")
plt.ylabel(r"Normalized $s_z(t)$")
plt.ylim(state_min - 0.05, state_max + 0.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Continue a real trajectory after observing its first third

The GRU is first warmed with the exact observed prefix. Starting from the final observed state, the remainder is generated autoregressively and compared with the held-out continuation.

In [ ]:
@torch.no_grad()
def continue_from_real_prefix(
    model,
    real_trajectory,
    prefix_length,
    time_end=1.0,
    temperature=1.0,
    generator=None,
):
    """Keep a real prefix and sample every state after it."""
    model.eval()
    device = next(model.parameters()).device
    real = torch.as_tensor(real_trajectory, dtype=torch.float32, device=device).flatten()

    if not 2 <= prefix_length < real.numel():
        raise ValueError("prefix_length must be between 2 and trajectory_length - 1.")

    source_times = torch.linspace(
        0.0,
        time_end,
        steps=real.numel(),
        device=device,
    )[:-1]

    # Warm the hidden state using exact source states before the last
    # observed state. The last observed state is then used to predict the
    # first unknown state without being processed twice.
    warm_features = torch.stack(
        (real[: prefix_length - 1], source_times[: prefix_length - 1]),
        dim=-1,
    ).unsqueeze(0)
    _, hidden = model(warm_features)

    current_sz = real[prefix_length - 1]
    generated_tail = []

    for step in range(prefix_length - 1, real.numel() - 1):
        features = torch.stack((current_sz, source_times[step])).reshape(1, 1, 2)
        logits, hidden = model(features, hidden)
        current_sz = model.sample_next_state(
            logits[0, 0],
            temperature=temperature,
            generator=generator,
        )
        generated_tail.append(current_sz)

    return torch.cat((real[:prefix_length], torch.stack(generated_tail)))


continuation_index = test_set.indices[0]
real_trajectory = dataset.sz[continuation_index]
prefix_length = len(real_trajectory) // 3
continuation_generator = torch.Generator(device=device).manual_seed(17)

continued_trajectory = continue_from_real_prefix(
    model,
    real_trajectory,
    prefix_length=prefix_length,
    time_end=time_end,
    temperature=1.0,
    generator=continuation_generator,
).cpu().numpy()

real_np = real_trajectory.cpu().numpy()
continuation_time = np.linspace(0.0, 70.0, len(real_np))
split_time = continuation_time[prefix_length - 1]
continuation_mae = np.mean(
    np.abs(continued_trajectory[prefix_length:] - real_np[prefix_length:])
)

print(f"Observed points: {prefix_length}/{len(real_np)}")
print(f"Continuation MAE for this stochastic sample: {continuation_mae:.6f}")
print(
    "Generated continuation range:",
    continued_trajectory[prefix_length:].min(),
    continued_trajectory[prefix_length:].max(),
)

plt.figure(figsize=(12, 4))
plt.plot(
    continuation_time,
    real_np,
    "k--",
    lw=1.8,
    label="complete real trajectory",
)
plt.plot(
    continuation_time[:prefix_length],
    real_np[:prefix_length],
    color="tab:green",
    lw=2.5,
    label="observed real first third",
)
plt.plot(
    continuation_time[prefix_length - 1 :],
    continued_trajectory[prefix_length - 1 :],
    color="tab:blue",
    lw=2,
    marker="o",
    ms=3,
    label="generated continuation",
)
plt.axvline(split_time, color="tab:red", ls=":", label="generation starts")
plt.xlabel(r"Time $t$")
plt.ylabel(r"Normalized $s_z(t)$")
plt.ylim(state_min - 0.05, state_max + 0.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Generate a held-out-size ensemble

This compares mean curves, spread, and physical support. Staying in range is guaranteed; matching the held-out mean and spread must still be checked empirically.

In [ ]:
test_indices = torch.as_tensor(test_set.indices, dtype=torch.long)
exact_test = dataset.sz[test_indices].to(device)
ensemble_generator = torch.Generator(device=device).manual_seed(seed + 1)

generated_test = generate_trajectories(
    model,
    initial_sz=exact_test[:, 0],
    n_time_points=exact_test.shape[1],
    time_end=time_end,
    temperature=1.0,
    generator=ensemble_generator,
)

summary = rollout_summary(generated_test, exact_test)
for name, value in summary.items():
    print(f"{name}: {value:.6f}")

generated_np = generated_test.cpu().numpy()
exact_np = exact_test.cpu().numpy()

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(physical_time, generated_np.mean(axis=0), lw=2, label="generated mean")
axes[0].plot(physical_time, exact_np.mean(axis=0), "k--", lw=2, label="exact test mean")
axes[0].set_ylabel(r"Mean $s_z(t)$")
axes[0].set_ylim(state_min - 0.05, state_max + 0.05)
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].fill_between(
    physical_time,
    np.percentile(generated_np, 10, axis=0),
    np.percentile(generated_np, 90, axis=0),
    alpha=0.3,
    label="generated 10–90%",
)
axes[1].plot(
    physical_time,
    np.percentile(exact_np, 10, axis=0),
    "k--",
    lw=1.5,
    label="exact 10%",
)
axes[1].plot(
    physical_time,
    np.percentile(exact_np, 90, axis=0),
    "k--",
    lw=1.5,
    label="exact 90%",
)
axes[1].set_xlabel(r"Time $t$")
axes[1].set_ylabel(r"Normalized $s_z(t)$")
axes[1].set_ylim(state_min - 0.05, state_max + 0.05)
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## Inspect a predicted grid distribution

This plot is useful for detecting an overly broad distribution, collapsed single-bin predictions, or probability accumulating at the boundaries.

In [ ]:
model.eval()
with torch.no_grad():
    example_features, _ = dataset[reference_index]
    example_logits, _ = model(example_features.unsqueeze(0).to(device))
    step = 0  # Change this to inspect another transition.
    probability = torch.softmax(example_logits[0, step], dim=-1).cpu().numpy()
    state_grid = model.state_grid.cpu().numpy()

plt.figure(figsize=(9, 3))
plt.plot(state_grid, probability)
plt.axvline(sz[reference_index, step + 1], color="black", ls="--", label="exact next state")
plt.xlabel(r"Candidate next $s_z$")
plt.ylabel("Probability")
plt.grid(alpha=0.3)
plt.legend()
plt.show()